In [ ]:
import re
import requests
import pandas as pd

NAVER_CLIENT_ID = ""
NAVER_CLIENT_SECRET = ""

NAVER_BLOG_API_URL = "https://naverapihub.apigw.ntruss.com/search/v1/blog"

headers = {
    "X-NCP-APIGW-API-KEY-ID": NAVER_CLIENT_ID,
    "X-NCP-APIGW-API-KEY": NAVER_CLIENT_SECRET
}

# 유틸리티 함수
def clean_text(text) :
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"&amp;", "", text)
    return text.strip()


brand_names = product_df["brand_name"].dropna().drop_duplicates().tolist()
# brand_names = brand_names[:1]

print("검색할 전체 브랜드 수:", len(brand_names))

# 블로그 컨텐츠 중 취합할 요소만 찾아서 컨텐츠별 저장할 딕셔너리를 모아놓기위한 용도의 리스트
all_blog_posts = list() # []

for brand_index, brand_name in enumerate(brand_names, 1) :
    print( f"[{brand_index}/{len(brand_names)}] {brand_name} 블로그 검색 중...")
    params = {
        "query": brand_name,
        "display": 10,
        "start": 1,
        "sort": "date",
        "format": "json"
    }
    try :
        response = requests.get(NAVER_BLOG_API_URL, headers=headers, params=params, timeout=10)

        if response.status_code == 200 :
            data = response.json()
            items = data.get("items", [])

            for item in items :
                blog_data = {
                    "search_brand": brand_name,
                    "title": clean_text(item.get("title", "")),
                    "link": item.get("link", ""),
                    "description": clean_text(item.get("description", "")),
                    "bloggername": item.get("bloggername", ""),
                    "bloggerlink": item.get("bloggerlink", ""),
                    "postdate": item.get("postdate", "")
                }
                
                all_blog_posts.append(blog_data)
                
    except Exception as e :
        print(f"API 호출 중 오류 발생 : {e}")
        
blog_df = pd.DataFrame(all_blog_posts)

blog_df["postdate"]= pd.to_datetime(blog_df["postdate"], format="%Y%m%d", errors="coerce") #NaT = Not a Time

print("수집 완료")
print("브랜드 수:", len(brand_names))
print("블로그 게시물 수:", len(blog_df))

In [ ]:
import re
import requests
import pandas as pd
import time

NAVER_CLIENT_ID = ""
NAVER_CLIENT_SECRET = ""

NAVER_BLOG_API_URL = (
    "https://naverapihub.apigw.ntruss.com/search/v1/blog"
)

headers = {
    "X-NCP-APIGW-API-KEY-ID": NAVER_CLIENT_ID,
    "X-NCP-APIGW-API-KEY": NAVER_CLIENT_SECRET
}

# ============================================================
# 텍스트 정제 함수
# ============================================================

def clean_text(text):
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"&amp;", "&", text)
    return text.strip()

# ============================================================
# 브랜드 목록
# ============================================================

brand_names = (
    product_df["brand_name"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

print("검색할 전체 브랜드 수:", len(brand_names))

# ============================================================
# 전체 블로그 데이터 저장
# ============================================================

all_blog_posts = []

# ============================================================
# 브랜드별 수집
# ============================================================

for brand_index, brand_name in enumerate(brand_names, 1):
    print(f"[{brand_index}/{len(brand_names)}] {brand_name} 수집 시작")

    # ========================================================
    # 브랜드당 최대 1,000개
    #
    # 1
    # 101
    # 201
    # ...
    # 901
    # ========================================================

    for start in range(1, 1001, 100):
        print(f"start={start} 검색 중...")
        params = {
            "query": brand_name,
            "display": 100, # 한 번 호출에 최대 100개
            "start": start,  # 1, 101, 201 ...
            "sort": "date",
            "format": "json"
        }

        try:
            response = requests.get(
                NAVER_BLOG_API_URL,
                headers=headers,
                params=params,
                timeout=10
            )
            
            # ------------------------------------------------
            # 정상 응답
            # ------------------------------------------------

            if response.status_code == 200:
                data = response.json()
                items = data.get("items", [])

                # 검색 결과가 더 이상 없으면 종료
                if len(items) == 0:
                    print("더 이상 검색 결과 없음")
                    break

                for item in items:

                    blog_data = {
                        "search_brand": brand_name,
                        "title": clean_text(item.get("title", "")),
                        "link": item.get("link",""),
                        "description": clean_text(item.get("description","")),
                        "bloggername": item.get("bloggername", ""),
                        "bloggerlink":item.get("bloggerlink", ""),
                        "postdate":item.get("postdate", "")
                    }

                    all_blog_posts.append(blog_data)

                print(f"{len(items)}개 수집")

            # ------------------------------------------------
            # API 오류
            # ------------------------------------------------

            else:
                print(f"API 오류: {response.status_code}")
                print(response.text)
                break


        except Exception as e:
            print(f"API 호출 중 오류 발생: {e}")
            break

        # 너무 연속적으로 요청하지 않도록 약간의 간격
        time.sleep(0.1)

# ============================================================
# DataFrame 생성
# ============================================================

blog_df = pd.DataFrame(all_blog_posts)

# ============================================================
# 날짜 형식 변환
# ============================================================

if not blog_df.empty:
    blog_df["postdate"] = pd.to_datetime(
        blog_df["postdate"],
        format="%Y%m%d",
        errors="coerce"
    )


# ============================================================
# 중복 URL 제거
#
# 같은 글이 여러 검색 결과에서 중복될 수 있으므로
# 필요할 경우 사용
# ============================================================

# blog_df = (
#     blog_df
#     .drop_duplicates(
#         subset=[
#             "search_brand",
#             "link"
#         ]
#     )
#     .reset_index(
#         drop=True
#     )
# )


# ============================================================
# 최종 결과
# ============================================================

print("=" * 70)
print("수집 완료")
print("브랜드 수:", len(brand_names))
print("블로그 게시물 수:", len(blog_df))
print("=" * 70)